# Notebook 09: Flow Matching 2D 玩具实验

**目标**：在 moons 数据集上训练 Rectified Flow，与 DDPM 对比少步采样质量

**前置**：L12, derive_07

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

x_data, _ = make_moons(5000, noise=0.05)
x_data = torch.tensor(x_data, dtype=torch.float32).to(device)
print(f'data: {x_data.shape}, mean={x_data.mean(0)}, std={x_data.std(0)}')

## 1. Velocity 网络（同 nb01 的 MLP 但输出 velocity）

In [ ]:
class VelocityNet(nn.Module):
    def __init__(self, dim_t=64, hidden=128):
        super().__init__()
        self.dim_t = dim_t
        self.net = nn.Sequential(
            nn.Linear(2 + dim_t, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),
        )
    def time_emb(self, t):
        half = self.dim_t // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device) / half)
        a = t[:, None].float() * freqs[None]
        return torch.cat([a.sin(), a.cos()], dim=-1)
    def forward(self, x, t):
        return self.net(torch.cat([x, self.time_emb(t)], dim=-1))

## 2. Rectified Flow 训练

Loss = MSE(v_pred, x_1 - x_0)，其中 x_t = (1-t) x_0 + t x_1

In [ ]:
model_fm = VelocityNet().to(device)
opt = torch.optim.Adam(model_fm.parameters(), lr=2e-3)

losses_fm = []
for step in range(3000):
    idx = torch.randint(0, len(x_data), (512,))
    x_1 = x_data[idx]
    x_0 = torch.randn_like(x_1)
    t = torch.rand(x_1.shape[0], device=device)
    x_t = (1 - t).unsqueeze(-1) * x_0 + t.unsqueeze(-1) * x_1
    target = x_1 - x_0
    v_pred = model_fm(x_t, t * 1000)  # 用 0-1000 作为 t embedding 输入更稳
    loss = F.mse_loss(v_pred, target)
    opt.zero_grad(); loss.backward(); opt.step()
    losses_fm.append(loss.item())
    if step % 500 == 0:
        print(f'step {step}: loss={loss.item():.4f}')

In [ ]:
@torch.no_grad()
def fm_sample(n_steps=10, n_samples=2000, method='euler'):
    model_fm.eval()
    x = torch.randn(n_samples, 2, device=device)
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = torch.full((n_samples,), i * dt, device=device)
        v = model_fm(x, t * 1000)
        if method == 'euler':
            x = x + v * dt
        elif method == 'heun':
            x_pred = x + v * dt
            t_next = torch.full((n_samples,), (i+1) * dt, device=device)
            v_next = model_fm(x_pred, t_next * 1000)
            x = x + (v + v_next) / 2 * dt
    return x

## 3. 少步采样质量对比

In [ ]:
step_counts = [1, 2, 5, 10, 50]
fig, axes = plt.subplots(1, len(step_counts), figsize=(3*len(step_counts), 3))
for ax, N in zip(axes, step_counts):
    torch.manual_seed(42)
    s = fm_sample(n_steps=N).cpu()
    ax.scatter(s[:, 0], s[:, 1], s=2, alpha=0.4)
    ax.scatter(x_data[:500, 0].cpu(), x_data[:500, 1].cpu(), s=1, alpha=0.2, c='r')
    ax.set_title(f'FM {N} steps'); ax.set_aspect('equal')
    ax.set_xlim(-2, 2.5); ax.set_ylim(-1.5, 2)
plt.tight_layout(); plt.show()

## 观察

FM 1-2 步质量已经不错（DDPM 通常 5 步以下完全失败）。原因：linear path 几乎是直线，少步 Euler 误差小。

## 4. 与 DDPM 对比（同等设置）

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T).to(device)
alphas = 1 - betas
ac = alphas.cumprod(0)

model_ddpm = VelocityNet().to(device)  # 复用同样的 MLP，但训 noise prediction
opt = torch.optim.Adam(model_ddpm.parameters(), lr=2e-3)
for step in range(3000):
    idx = torch.randint(0, len(x_data), (512,))
    x_0 = x_data[idx]
    t = torch.randint(0, T, (512,), device=device)
    eps = torch.randn_like(x_0)
    x_t = ac[t].sqrt().unsqueeze(-1) * x_0 + (1-ac[t]).sqrt().unsqueeze(-1) * eps
    loss = F.mse_loss(model_ddpm(x_t, t), eps)
    opt.zero_grad(); loss.backward(); opt.step()
print('DDPM trained')

In [ ]:
@torch.no_grad()
def ddim_sample(n_steps=50, n_samples=2000):
    model_ddpm.eval()
    x = torch.randn(n_samples, 2, device=device)
    step = T // n_steps
    ts = list(range(0, T, step))[::-1]
    for i, ti in enumerate(ts):
        t_prev = ts[i+1] if i+1 < len(ts) else -1
        t_tensor = torch.full((n_samples,), ti, device=device, dtype=torch.long)
        eps = model_ddpm(x, t_tensor)
        x0 = (x - (1-ac[ti]).sqrt() * eps) / ac[ti].sqrt()
        x0 = x0.clamp(-3, 3)
        if t_prev < 0:
            x = x0
        else:
            x = ac[t_prev].sqrt() * x0 + (1 - ac[t_prev]).sqrt() * eps
    return x

fig, axes = plt.subplots(2, len(step_counts), figsize=(3*len(step_counts), 6))
for j, N in enumerate(step_counts):
    torch.manual_seed(42)
    s_fm = fm_sample(n_steps=N).cpu()
    axes[0][j].scatter(s_fm[:, 0], s_fm[:, 1], s=2, alpha=0.4)
    axes[0][j].scatter(x_data[:500, 0].cpu(), x_data[:500, 1].cpu(), s=1, alpha=0.2, c='r')
    axes[0][j].set_title(f'FM {N} steps'); axes[0][j].set_aspect('equal')
    axes[0][j].set_xlim(-2, 2.5); axes[0][j].set_ylim(-1.5, 2)
    torch.manual_seed(42)
    s_d = ddim_sample(n_steps=N).cpu()
    axes[1][j].scatter(s_d[:, 0], s_d[:, 1], s=2, alpha=0.4)
    axes[1][j].scatter(x_data[:500, 0].cpu(), x_data[:500, 1].cpu(), s=1, alpha=0.2, c='r')
    axes[1][j].set_title(f'DDIM {N} steps'); axes[1][j].set_aspect('equal')
    axes[1][j].set_xlim(-2, 2.5); axes[1][j].set_ylim(-1.5, 2)
plt.tight_layout(); plt.show()

## 思考题

1. 实现 Heun 二阶 sampler，与 Euler 比较 NFE-质量
2. 把 FM target 从 `x_1 - x_0` 改成 `x_0` prediction，重写 sampler，观察差异
3. 实现 "reflow"：用训好的 FM 生成 (x_0, x_1_pred) pairs，用这些重训
4. 在更复杂的 2D 数据（如 25 个 Gaussian mixture）上对比 FM vs DDPM